출처:https://github.com/Gepetto/supaero2025/blob/main/1-forward_geom.ipynb

In [39]:
import time
import numpy as np
from scipy.optimize import fmin_bfgs,fmin_slsqp
from supaero2025.meshcat_viewer_wrapper import MeshcatVisualizer
from supaero2025.meshcat_viewer_wrapper.transformations import planar, translation2d
from numpy.linalg import norm,inv,pinv,svd,eig

In [40]:
viz = MeshcatVisualizer()
viz.viewer.jupyter_cell()

You can open the visualizer by visiting the following URL:
http://127.0.0.1:7013/static/


# 1. Create a 2D robot

In [41]:
viz.addSphere('joint1',.1,[1,0,0,1])
viz.addSphere('joint2',.1,[1,0,0,1])
viz.addSphere('joint3',.1,[1,0,0,1])
viz.addCylinder('arm1',.75,.05,[.65,.65,.65,1])
viz.addCylinder('arm2',.75,.05,[.65,.65,.65,1])
viz.addSphere('target',.1001,[0,.8,.1,1])

In [7]:
q = np.random.rand(2) * 6 - 3

In [8]:
q

array([2.38050135, 0.28325679])

In [9]:
def display(q):
    '''Display the robot in Gepetto Viewer. '''
    assert (q.shape == (2,))
    c0 = np.cos(q[0])
    s0 = np.sin(q[0])
    c1 = np.cos(q[0] + q[1])
    s1 = np.sin(q[0] + q[1])
    viz.applyConfiguration('joint1',planar(0,           0,           0))
    viz.applyConfiguration('arm1'  ,planar(c0 / 2,      s0 / 2,      q[0]))
    viz.applyConfiguration('joint2',planar(c0,          s0,          q[0]))
    viz.applyConfiguration('arm2'  ,planar(c0 + c1 / 2, s0 + s1 / 2, q[0] + q[1]))
    viz.applyConfiguration('joint3',planar(c0 + c1,     s0 + s1,     q[0] + q[1]))


In [10]:
display(q) # Display the robot in the viewer

# 2. Optimize the configuration

In [42]:
target = np.array([.5, .5])
viz.applyConfiguration('target',translation2d(target[0],target[1]))

In [43]:
def endeffector(q):
    x,y = 1 * np.cos(q[0]) + 1 * np.cos(q[0] + q[1]), \
            1 * np.sin(q[0]) + 1 * np.sin(q[0] + q[1])
    return np.array([x,y])
                            
def cost(q):
    eff = endeffector(q)
    return norm(eff - target)**2

In [44]:
def callback(q):
    display(q)
    time.sleep(.5)

In [45]:
q0 = np.array([0.0, 0.0])
qopt_bfgs = fmin_bfgs(cost, q0, callback=callback)
print('\n *** Optimal configuration from BFGS = %s \n\n\n\n' % qopt_bfgs)

Optimization terminated successfully.
         Current function value: 0.000000
         Iterations: 12
         Function evaluations: 45
         Gradient evaluations: 15

 *** Optimal configuration from BFGS = [-0.42403109  2.41885841] 






# 3. What configuration to optimize?




In [46]:
x1, y1, th1, x2, y2, th2, x3, y3, th3 = q0 = np.zeros(9)

In [47]:
def endeffector_9(ps):
    assert (ps.shape == (9, ))
    x1, y1, t1, x2, y2, t2, x3, y3, t3 = ps
    return np.array([x3, y3])

In [48]:
def display_9(ps):
    '''Display the robot in the Viewer. '''
    assert (ps.shape == (9, ))
    x1, y1, t1, x2, y2, t2, x3, y3, t3 = ps
    viz.applyConfiguration('joint1',planar(x1,                  y1,                  t1))
    viz.applyConfiguration('arm1'  ,planar(x1 + np.cos(t1) / 2, x1 + np.sin(t1) / 2, t1))
    viz.applyConfiguration('joint2',planar(x2,                  y2,                  t2))
    viz.applyConfiguration('arm2'  ,planar(x2 + np.cos(t2) / 2, y2 + np.sin(t2) / 2, t2))
    viz.applyConfiguration('joint3',planar(x3,                  y3,                  t3))


In [49]:
def cost_9(ps):
    eff = endeffector_9(ps)
    return norm(eff - target)**2

## Working under constraints

In [56]:
qrand9 = np.random.rand(9)
print(qrand9)
display_9(qrand9)

[0.46453876 0.85821347 0.22251811 0.41757214 0.94329571 0.37954724
 0.34656625 0.38298931 0.7196038 ]


In [57]:
def constraint_9(q):
    constraints = np.zeros(6) # Fill me
    x1, y1, t1, x2, y2, t2, x3, y3, t3 = q

    constraints[0] = x2- x1 - np.cos(t1)
    constraints[1] = y2- y1 - np.sin(t1)
    constraints[2] = x3- x2 - np.cos(t2)
    constraints[3] = y3- y2 - np.sin(t2)
    constraints[4] = x1
    constraints[5] = y1

    return constraints

In [58]:
print(cost_9(qrand9), constraint_9(qrand9))

0.03723341551993217 [-1.02231145 -0.1356041  -0.99983836 -0.93080637  0.46453876  0.85821347]


In [59]:
def callback_9(ps):
    display_9(ps)
    time.sleep(.5)

## Solve with a penalty cost

In [60]:
def penalty(ps):
    return cost_9(ps) + 100 * sum(np.square(constraint_9(ps)))

In [61]:
qopt = fmin_bfgs(penalty, qrand9, callback=callback_9)

         Current function value: 0.000000
         Iterations: 50
         Function evaluations: 1090
         Gradient evaluations: 108


/home/home/anaconda3/envs/pino/lib/python3.12/site-packages/scipy/optimize/_optimize.py:1330: OptimizeWarning: Desired error not necessarily achieved due to precision loss.
  res = _minimize_bfgs(f, x0, args, fprime, callback=callback, **opts)
